Fixed smoke test to use SCORING_URL_INTERNAL variable defined by the deploy cell
*Co-authored with CoCo*

# Notebook 4: Deploy SPCS Fraud Scoring Service

Builds and deploys a custom Docker container to Snowpark Container Services (SPCS).
The scoring service implements the 3-thread payment gateway pattern:

```
POST /score  (public HTTPS ingress)
     |
     |-- Thread B: OFS ingest  (async, internal URL — updates velocity features)
     '-- Thread C: OFS query x5 parallel  (sync, internal URL — fetches features)
                   -> compute derived features (~0.1ms)
                   -> XGBoost predict (~5ms)
                   -> return {score, decision, timing}
```

OFS internal URLs (`*.svc.spcs.internal`) are used because the container runs
**inside** the SPCS cluster — the same cluster as the OFS runtime.
This gives ~2-5ms feature lookup vs ~15-30ms via the public URL.

**Prerequisites:** Run nb01, nb02, nb03 first. Docker Desktop must be running locally.


In [ ]:
import os, json, time, requests
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql('USE ROLE FRAUD_MLOPS').collect()
session.sql('USE DATABASE FRAUD_DEMO_PROD').collect()
session.sql('USE SCHEMA ML').collect()
session.sql('USE WAREHOUSE FRAUD_DEMO_WH').collect()

# Session token — auto-derived from active Notebooks session.
# Used for REST calls to the scoring service endpoint.
PAT = session.connection.rest.token or os.environ.get('SNOWFLAKE_PAT', '')
if not PAT:
    raise RuntimeError('Could not obtain session token.')
print(f'Session: FRAUD_DEMO_PROD.ML')
print(f'Auth token: {PAT[:16]}...')


## 4.1 Get OFS Internal URLs

Retrieve the SPCS-internal OFS endpoint URLs to inject into the container spec.
These URLs are only resolvable from within the SPCS cluster — exactly where the container runs.

In [ ]:
# Get OFS internal URLs — these are injected into the container spec.
# The container runs inside SPCS where internal_url (*.svc.spcs.internal) is resolvable.
import json as _json

_raw = session.sql(
    "SELECT SYSTEM$GET_FEATURE_STORE_ONLINE_SERVICE_STATUS('FRAUD_DEMO_DEV.FEATURE_STORE')"
).collect()[0][0]
_svc       = _json.loads(_raw)
_endpoints = _svc.get('endpoints', [])

def _pick_url(ep_name, prefer_internal=False):
    """Extract URL from endpoints list or dict. Handles both formats."""
    if isinstance(_endpoints, list):
        # SDK returns a list: [{"name": "ingest", "url": "...", "internal_url": "..."}]
        for ep in _endpoints:
            if ep.get('name') == ep_name:
                if prefer_internal:
                    url = ep.get('internal_url') or ep.get('url', '')
                else:
                    url = ep.get('url') or ep.get('internal_url', '')
                return url.rstrip('/')
        raise RuntimeError(f'Endpoint {ep_name!r} not found. Available: {[e.get("name") for e in _endpoints]}')
    # Dict format fallback
    ep = _endpoints.get(ep_name, _endpoints.get(ep_name.lower(), {}))
    if isinstance(ep, str): return ep.rstrip('/')
    if prefer_internal:
        url = ep.get('internal_url') or ep.get('internal') or ep.get('url', '')
    else:
        url = ep.get('url') or ep.get('public') or ep.get('internal_url', '')
    if not url:
        raise RuntimeError(f'No URL for {ep_name!r}. Available: {json.dumps(_endpoints, indent=2)}')
    return url.rstrip('/')

OFS_INGEST_INTERNAL = _pick_url('ingest', prefer_internal=True)
OFS_QUERY_INTERNAL  = _pick_url('query',  prefer_internal=True)

print('OFS endpoints (internal — for SPCS container):')
print(f'  Ingest: {OFS_INGEST_INTERNAL}')
print(f'  Query:  {OFS_QUERY_INTERNAL}')


## 4.2 Build and Push Docker Image

Run the commands below **in your local terminal** (not in this notebook).
Docker Desktop must be running. The image is ~250MB.

The cell below prints the exact commands to copy-paste.


In [ ]:
# Get image repository URL from Snowflake
repo_rows = session.sql('SHOW IMAGE REPOSITORIES IN SCHEMA FRAUD_DEMO_PROD.ML').collect()
if not repo_rows:
    raise RuntimeError('Image repository not found. Run setup.sql first.')
REPO_URL  = repo_rows[0]['repository_url']
IMAGE_TAG = f'{REPO_URL}/fraud_scorer:latest'

# Snowflake registry login credentials
account = session.sql('SELECT CURRENT_ACCOUNT()').collect()[0][0].lower()
user    = session.sql('SELECT CURRENT_USER()').collect()[0][0]

print('Run these commands in your local terminal:')
print('=' * 60)
print(f'cd {os.path.dirname(os.path.abspath("."))}')
print(f'')
print(f'# Log in to Snowflake image registry')
print(f'snow spcs image-registry login')
print(f'')
print(f'# Build the scoring service image')
print(f'docker build -t {IMAGE_TAG} services/fraud_scorer/')
print(f'')
print(f'# Push to Snowflake registry')
print(f'docker push {IMAGE_TAG}')
print('=' * 60)
print(f'\nImage tag: {IMAGE_TAG}')
print('After pushing, continue to cell 4.3 to deploy the service.')


## 4.3 Deploy SPCS Service

In [ ]:
import yaml

# Load spec template and inject real values
import pathlib
for p in [pathlib.Path('/snowflake/stages/user__public__fraud_detection_ofs_/services/fraud_scorer/spec.yaml'),
          pathlib.Path('services/fraud_scorer/spec.yaml'),
          pathlib.Path('../services/fraud_scorer/spec.yaml')]:
    if p.exists():
        spec_path = p
        break

spec_text = spec_path.read_text()
spec_text = spec_text.replace('__IMAGE_URL__',          IMAGE_TAG)
spec_text = spec_text.replace('__OFS_INGEST_INTERNAL__', OFS_INGEST_INTERNAL)
spec_text = spec_text.replace('__OFS_QUERY_INTERNAL__',  OFS_QUERY_INTERNAL)

spec = yaml.safe_load(spec_text)
spec_json = json.dumps(spec)

# Drop existing service
session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql('DROP SERVICE IF EXISTS FRAUD_DEMO_PROD.ML.FRAUD_SCORING_SERVICE').collect()

print('Deploying FRAUD_SCORING_SERVICE...')
session.sql(f"""
    CREATE SERVICE FRAUD_DEMO_PROD.ML.FRAUD_SCORING_SERVICE
      IN COMPUTE POOL FRAUD_OFS_CPU_POOL
      FROM SPECIFICATION $${spec_json}$$
      MIN_INSTANCES = 1
      MAX_INSTANCES = 2
      COMMENT = 'Fraud scoring: OFS feature lookup + XGBoost inference'
""").collect()

print('Service created. Polling for RUNNING...')
t0 = time.time()
while True:
    svc_rows = session.sql(
        "SHOW SERVICES LIKE 'FRAUD_SCORING_SERVICE' IN SCHEMA FRAUD_DEMO_PROD.ML"
    ).collect()
    status = svc_rows[0]['status'] if svc_rows else 'UNKNOWN'
    elapsed = time.time() - t0
    if status == 'RUNNING':
        break
    if status in ('FAILED', 'DELETING'):
        # Show logs for debugging
        logs = session.sql("CALL SYSTEM$GET_SERVICE_LOGS('FRAUD_DEMO_PROD.ML.FRAUD_SCORING_SERVICE', 0, 'fraud-scorer', 50)").collect()
        print(f'Service logs: {logs[0][0] if logs else "none"}')
        raise RuntimeError(f'Service deploy failed: {status}')
    if elapsed > 300:
        raise RuntimeError(f'Timeout after 5min. Status: {status}')
    print(f'  [{elapsed:.0f}s] {status}')
    time.sleep(15)

# Get endpoint
ep_rows = session.sql("SHOW ENDPOINTS IN SERVICE FRAUD_DEMO_PROD.ML.FRAUD_SCORING_SERVICE").collect()
SCORING_DNS = svc_rows[0]['dns_name']
SCORING_URL_INTERNAL = f'http://{SCORING_DNS}:8080'

print(f'\nService RUNNING in {time.time()-t0:.0f}s')
print(f'Internal endpoint: {SCORING_URL_INTERNAL}')
if ep_rows:
    ingress = ep_rows[0].asDict().get('ingress_url', '')
    if ingress:
        print(f'Public endpoint:   https://{ingress}')

## 4.4 Smoke Test

In [ ]:
# Smoke test: verify the service is healthy and can score a transaction.
AUTH = {'Authorization': f'Snowflake Token="{PAT}"', 'Content-Type': 'application/json'}

# Health check
r = requests.get(f'{SCORING_URL_INTERNAL}/health', headers=AUTH, timeout=10)
r.raise_for_status()
health = r.json()
print(f'Health: {health}')

# Test score
test_txn = {
    'Cust_Ref':      'ZLCH00099999',
    'Merchant_Id':   'M_GBR_001',
    'Token_Ref':     'DPAN_TEST_001',
    'IP_Address':    '185.23.44.100',
    'Trans_Amount':  250.00,
    'Merch_Country': 'GBR',
    'Trans_DateTime': '2026-07-03T10:00:00Z',
}
r = requests.post(f'{SCORING_URL_INTERNAL}/score', headers=AUTH, json=test_txn, timeout=15)
r.raise_for_status()
result = r.json()

print(f'Score:    {result["score"]}')
print(f'Decision: {result["decision"]}')
print(f'Timing:   OFS {result["timing"]["ofs_query_ms"]}ms | '
      f'XGBoost {result["timing"]["xgb_ms"]}ms | '
      f'Total {result["timing"]["total_ms"]}ms')

# Warm-up loop: measure steady-state latency after connection pooling
print('\n--- Steady-state latency (5 runs) ---')
ofs_times = []
for i in range(5):
    r = requests.post(f'{SCORING_URL_INTERNAL}/score', headers=AUTH, json=test_txn, timeout=15)
    r.raise_for_status()
    t = r.json()['timing']
    ofs_times.append(t['ofs_query_ms'])
    print(f"  Run {i+1}: OFS {t['ofs_query_ms']}ms | XGBoost {t['xgb_ms']}ms | Total {t['total_ms']}ms")

print(f'\n  Avg OFS (steady-state): {sum(ofs_times)/len(ofs_times):.1f}ms')
print(f'\nSCORING_URL_INTERNAL = "{SCORING_URL_INTERNAL}"  # copy this for nb05 and nb06')